# AquaSense AI  :  Sprint 4 Colab : amélioration MLP (tuning)

**Équipe :** EHTP MIG S4

Tuning complet : MLP + ResidualMLP + 1D-CNN + grid search.

**Prérequis :** `AquaSense_S4_Colab.zip` sur Drive  -  Runtime **GPU T4**  -  ~45-90 min

**Baseline précédent :** F1-Macro 0.53  -  Recall needs repair 0.60

## 1. Setup (première fois)

Exécute les cellules **1.1** puis **1.2**, puis **Runtime  ->  Restart session**.

> **Ne pas** faire `%pip install numpy` ou `pandas` seuls sur Colab  :  ça casse TensorFlow (`_blas_supports_fpe`).
> On installe seulement `imbalanced-learn`.

In [1]:
# 1.1  :  Monter Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# 1.2  :  Dézip + pip MINIMAL (pas de upgrade numpy/pandas !)
ZIP = "/content/drive/MyDrive/AquaSense_S4_Colab.zip"  # ajuste si besoin

!unzip -o -q "{ZIP}" -d /content/
%cd /content/AquaSense_S4_Colab

%pip install -q imbalanced-learn

!ls -la data/cleaned/train_clean.csv src/train_dl.py

/content/AquaSense_S4_Colab
-rw-rw-rw- 1 root root 16353177 Jun 18 22:22 data/cleaned/train_clean.csv
-rw-rw-rw- 1 root root     8588 Jun 19 02:15 src/train_dl.py


### STOP  :  Restart session

Menu : **Runtime  ->  Restart session** (ou Ctrl+M puis `.`)

Ensuite passe à la section **2** (ne ré-exécute pas le `%pip`).

## 2. Après restart  :  vérification GPU

Exécute **2.1** puis **2.2**.

In [3]:
# 2.1  :  Remonter Drive + aller dans le projet (SANS pip)
from google.colab import drive
drive.mount("/content/drive")

%cd /content/AquaSense_S4_Colab

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/AquaSense_S4_Colab


In [4]:
# 2.2  :  Vérifier TensorFlow + GPU
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import tensorflow as tf

PROJECT_ROOT = Path("/content/AquaSense_S4_Colab")
sys.path.insert(0, str(PROJECT_ROOT))

print("numpy  :", np.__version__)
print("scipy  :", scipy.__version__)
print("pandas :", pd.__version__)
print("tf     :", tf.__version__)
print("GPU    :", tf.config.list_physical_devices("GPU"))
print("train_dl :", (PROJECT_ROOT / "src/train_dl.py").exists())

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Pas de GPU  :  Runtime -> Change runtime type -> T4 GPU")

numpy  : 2.0.2
scipy  : 1.16.3
pandas : 2.2.2
tf     : 2.20.0
GPU    : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
train_dl : True


### Si erreur `_blas_supports_fpe` (numpy/scipy cassés)

Exécute la cellule **2.3 FIX** ci-dessous, puis **Restart session**, puis refais **2.1** et **2.2** seulement.

In [5]:
# 2.3 FIX  :  seulement si import tensorflow echoue (numpy/scipy desynchronises)
%pip install -q --upgrade scipy numpy
print("OK  :  maintenant Runtime -> Restart session, puis section 2.1 et 2.2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 104.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
OK  :  maintenant Runtime -> Restart session, puis section 2.1 et 2.2


## 3. Tuning complet (~45-90 min)

`python -m src.train_dl tune`  ->  MLP + Residual + CNN + grid search

In [6]:
%cd /content/AquaSense_S4_Colab
!python -m src.train_dl tune

/content/AquaSense_S4_Colab

=== MLP baseline ===
2026-06-19 02:38:57.949704: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1781836737.951174    3657 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Epoch 1/100
I0000 00:00:1781836744.242947    3751 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.4230 - loss: 1.1223 - val_accuracy: 0.3312 - val_loss: 1.1159 - learning_rate: 0.0010
Epoch 2/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4974 - loss: 1.0010 - val_accuracy: 0.5604 - val_loss: 0.9689 - learning_rate: 0.0010
Epoch 3/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - 

## 4. Résultats

In [7]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path("/content/AquaSense_S4_Colab")
csv_path = ROOT / "reports/sprint_04_dl_comparison.csv"
metrics_path = ROOT / "reports/sprint_04_metrics.json"

if csv_path.exists():
    display(pd.read_csv(csv_path).sort_values("f1_macro", ascending=False))

if metrics_path.exists():
    results = json.loads(metrics_path.read_text(encoding="utf-8"))
    champ = results.get("champion_dl", "?")
    m = results["models"].get(champ, {})
    f1, recall, acc = m.get("f1_macro", 0), m.get("recall_needs_repair", 0), m.get("accuracy", 0)
    print(f"Champion DL : {champ}")
    print(f"  F1-Macro            : {f1:.4f}  ({'OK' if f1 >= 0.72 else 'NON'} vs 0.72)")
    print(f"  Recall needs repair : {recall:.4f}  ({'OK' if recall >= 0.65 else 'NON'} vs 0.65)")
    print(f"  Accuracy            : {acc:.4f}")
    print("\nRef ML S3 : RF F1=0.6658 | XGB recall=0.6952 | MLP baseline=0.5297")

,architecture,f1_macro,f1_needs_repair,recall_needs_repair,accuracy
0,mlp_l2_0.001,0.540980,0.311284,0.602549,0.609933
1,mlp_baseline,0.529678,0.299174,0.566628,0.603788
2,residual_mlp,0.527596,0.274304,0.302433,0.634091
3,mlp_l2_0.0,0.527170,0.297273,0.574739,0.598401
4,cnn1d,0.411280,0.172566,0.225956,0.506650
5,mlp_tuned,0.381712,0.206480,0.930475,0.380303


Champion DL : mlp_baseline
  F1-Macro            : 0.5297  (NON vs 0.72)
  Recall needs repair : 0.5666  (NON vs 0.65)
  Accuracy            : 0.6038

Ref ML S3 : RF F1=0.6658 | XGB recall=0.6952 | MLP baseline=0.5297


## 5. Copier sur Drive

In [8]:
import shutil
from pathlib import Path

ROOT = Path("/content/AquaSense_S4_Colab")
DRIVE = Path("/content/drive/MyDrive/AquaSense_DL_results")
DRIVE.mkdir(parents=True, exist_ok=True)

files = [
    "mlp_tuned_best_v1.keras", "best_dl_model.keras",
    "residual_mlp_v1.keras", "cnn1d_v1.keras", "mlp_best_v1.keras",
    "sprint_04_metrics.json", "sprint_04_dl_comparison.csv",
]
for name in files:
    src = (ROOT / "models" / name) if name.endswith(".keras") else (ROOT / "reports" / name)
    if src.exists():
        shutil.copy(src, DRIVE / name)
        print("OK ->", DRIVE / name)
    else:
        print("(absent)", name)
print("\nDossier:", DRIVE)

OK -> /content/drive/MyDrive/AquaSense_DL_results/mlp_tuned_best_v1.keras
OK -> /content/drive/MyDrive/AquaSense_DL_results/best_dl_model.keras
OK -> /content/drive/MyDrive/AquaSense_DL_results/residual_mlp_v1.keras
OK -> /content/drive/MyDrive/AquaSense_DL_results/cnn1d_v1.keras
OK -> /content/drive/MyDrive/AquaSense_DL_results/mlp_best_v1.keras
OK -> /content/drive/MyDrive/AquaSense_DL_results/sprint_04_metrics.json
OK -> /content/drive/MyDrive/AquaSense_DL_results/sprint_04_dl_comparison.csv

Dossier: /content/drive/MyDrive/AquaSense_DL_results
